In [1]:
import os
from enum import Enum
from typing import Any

from dotenv import load_dotenv
from pydantic import SecretStr
from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI
from langchain_core.language_models import BaseChatModel

load_dotenv()

class Family(str, Enum):
    ANTHROPIC = "anthropic"
    OPENAI = "openai"
    DEEPSEEK = "deepseek"
    QWEN = "qwen"
    LOCAL = "local"



/home/rossimar26/pasantia carlos/NLP4RE/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class LLMFactory:
    @classmethod
    def create(
        cls,
        family: Family,
        model: str,
        provider_url: str | None = None, 
        api_key: str | None = None,
        temperature: float = 0,
        max_tokens: int = 1024,
        **kwargs: Any,
    ) -> BaseChatModel:

        if family == Family.ANTHROPIC:
            anthropic_kwargs: dict[str, Any] = dict(
                model_name=model,
                api_key=SecretStr(api_key or ""),
                temperature=temperature,
                max_tokens_to_sample=max_tokens,
                timeout=30,
                **kwargs,
            )
            if provider_url:
                anthropic_kwargs["base_url"] = provider_url
            return ChatAnthropic(**anthropic_kwargs)

        resolved_key = api_key if api_key else ("lm-studio" if family == Family.LOCAL else "")
        openai_kwargs: dict[str, Any] = dict(
            model=model,
            api_key=SecretStr(resolved_key),
            temperature=temperature,
            max_completion_tokens=max_tokens,
            **kwargs,
        )
        if provider_url:
            openai_kwargs["base_url"] = provider_url

        return ChatOpenAI(**openai_kwargs)

In [ ]:

llm = LLMFactory.create(
    family=Family.QWEN,
    model=os.getenv("MODEL_QWEN", "qwen/qwen3-coder:free"),
    provider_url=os.getenv("API_BASE_URL_QWEN", "https://openrouter.ai/api/v1"),
    api_key=os.getenv("API_KEY_QWEN","NONE"),
)

llm.invoke("quien eres?")

# Modelo de familia QWEN servido por OpenRouter
#llm = LLMFactory.create(
#    family=Family.QWEN,
#    model="qwen/qwen3-coder:free",
#    provider="https://openrouter.ai/api/v1",
#    api_key=os.getenv("OPENROUTER_API_KEY"),
#)

AIMessage(content='Soy Qwen, un asistente de inteligencia artificial creado por Alibaba Cloud. Estoy aquí para ayudarte con información, responder preguntas y auxiliarte en diversas tareas según sea necesario. ¿En qué puedo ayudarte hoy?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 34, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 6.46e-06, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 6.46e-06, 'upstream_inference_prompt_cost': 1.36e-06, 'upstream_inference_completions_cost': 5.1e-06}}, 'model_provider': 'openai', 'model_name': 'qwen/qwen-2.5-7b-instruct', 'system_fingerprint': None, 'id': 'gen-1776903489-lvN0DMQsWMSHJXiDkLa8', 'finish_reason': 'st

In [ ]:
from langchain_community.vectorstores.pgvector import PGVector
from langchain_openai import OpenAIEmbeddings

db_name = os.getenv("POSTGRES_DB")
db_password = os.getenv("POSTGRES_PASSWORD")
db_user = os.getenv("POSTGRES_USER")
db_host = os.getenv("POSTGRES_HOST", "localhost")
db_port = os.getenv("POSTGRES_PORT", "5432")

CONNECTION_STRING = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
COLLECTION_NAME = "iso_standards"



embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",  
    api_key=os.getenv("OPENAI_API_KEY")
)

llm = LLMFactory.create(
    family=Family.QWEN,
    model=os.getenv("MODEL_QWEN", "qwen/qwen3-coder:free"),
    provider_url=os.getenv("API_BASE_URL_QWEN", "https://openrouter.ai/api/v1"),
    api_key=os.getenv("API_KEY_QWEN"),
    temperature=0.1
)

vector_store = PGVector(
    connection_string=CONNECTION_STRING,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

import json
with open('./docs/rules.json', 'r', encoding='utf-8') as f: rules_data = json.load(f)
with open('./docs/examples.json', 'r', encoding='utf-8') as f: examples_data = json.load(f)

rules_str = ""
for r in rules_data:
    rules_str += f"- {r['typeC']}: {r['description']}\n"
    for level, desc in r['criterion'].items(): rules_str += f"  * Nivel {level}: {desc}\n"

examples_str = ""
for idx, ex in enumerate(examples_data):
    examples_str += f"Ejemplo {idx+1}:\nRequerimiento: {ex['text']}\nEvaluación:\n"
    for tag, score in ex['tags'].items(): examples_str += f" - {tag}: {score} ({ex['explanations'].get(tag, 'OK')})\n"
    examples_str += "\n"

template_rag = f"""Eres un experto en Ingeniería de Requisitos en una pasantía de investigación.
Se te proporcionará contexto normativo basado en la ISO 29148.
Tu tarea es recibir un requerimiento de un usuario, evaluarlo y sugerir cómo mejorarlo basándote en las siguientes reglas y ejemplos.

Reglas de evaluación:
{rules_str}

Ejemplos de cómo evaluar:
{examples_str}

Contexto normativo recuperado de la ISO:
{{context}}

Requerimiento propuesto por el usuario:
{{requirement}}

Instrucciones:
1. Analiza el requerimiento del usuario basándote en las métricas y reglas (VERIFIABILITY, ATOMICITY, AMBIGUITY, COMPLETENESS, TRACEABLE).
2. Da recomendaciones específicas para mejorarlo.
3. Presenta una versión reescrita y óptima del requerimiento."""

prompt_rag = ChatPromptTemplate.from_template(template_rag)

In [ ]:
from langchain_core.runnables import RunnableLambda

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

retriever_chain = retriever | RunnableLambda(format_docs)

In [ ]:
print(retriever)

In [ ]:
chain_rag = (
    {
        "context": retriever_chain,
        "requirement": RunnablePassthrough()
    }
    | prompt_rag
    | llm
    | StrOutputParser()
)
# hacer la prueba con un requerimiento deficiente e impreciso
requerimiento_usuario = "El sistema de base de datos deberá ser muy rápido y soportar bastantes cosas sin caerse."

print("Evaluando requerimiento...\n")
resultado = chain_rag.invoke(requerimiento_usuario)
print(resultado)